In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score,
    precision_score, f1_score, cohen_kappa_score,
    matthews_corrcoef
)
from imblearn.over_sampling import RandomOverSampler
import kagglehub
import shutil
import os
from ucimlrepo import fetch_ucirepo

In [2]:
# Kaggle
dataset_dir = "data"
if len(os.listdir(dataset_dir)) == 0:
    path = kagglehub.dataset_download("redwankarimsony/heart-disease-data")
    shutil.move(path, dataset_dir)

print("dataset downloaded")

# UCI dataset
# heart_disease = fetch_ucirepo(id=45) 
# X = heart_disease.data.features 
# y = heart_disease.data.targets 

dataset downloaded


In [3]:
# uncomment if use data from kaggle
data_path = "data/6/heart_disease_uci.csv"
df = pd.read_csv(data_path, index_col=0)
df.head(20)

,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
id,,,,,,,,,,,,,,,
1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0
6,56,Male,Cleveland,atypical angina,120.0,236.0,False,normal,178.0,False,0.8,upsloping,0.0,normal,0
7,62,Female,Cleveland,asymptomatic,140.0,268.0,False,lv hypertrophy,160.0,False,3.6,downsloping,2.0,normal,3
8,57,Female,Cleveland,asymptomatic,120.0,354.0,False,normal,163.0,True,0.6,upsloping,0.0,normal,0
9,63,Male,Cleveland,asymptomatic,130.0,254.0,False,lv hypertrophy,147.0,False,1.4,flat,1.0,reversable defect,2


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 920 entries, 1 to 920
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       920 non-null    int64  
 1   sex       920 non-null    object 
 2   dataset   920 non-null    object 
 3   cp        920 non-null    object 
 4   trestbps  861 non-null    float64
 5   chol      890 non-null    float64
 6   fbs       830 non-null    object 
 7   restecg   918 non-null    object 
 8   thalch    865 non-null    float64
 9   exang     865 non-null    object 
 10  oldpeak   858 non-null    float64
 11  slope     611 non-null    object 
 12  ca        309 non-null    float64
 13  thal      434 non-null    object 
 14  num       920 non-null    int64  
dtypes: float64(5), int64(2), object(8)
memory usage: 115.0+ KB


In [5]:
df.describe()

,age,trestbps,chol,thalch,oldpeak,ca,num
count,920.000000,861.000000,890.000000,865.000000,858.000000,309.000000,920.000000
mean,53.510870,132.132404,199.130337,137.545665,0.878788,0.676375,0.995652
std,9.424685,19.066070,110.780810,25.926276,1.091226,0.935653,1.142693
min,28.000000,0.000000,0.000000,60.000000,-2.600000,0.000000,0.000000
25%,47.000000,120.000000,175.000000,120.000000,0.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,140.000000,0.500000,0.000000,1.000000
75%,60.000000,140.000000,268.000000,157.000000,1.500000,1.000000,2.000000
max,77.000000,200.000000,603.000000,202.000000,6.200000,3.000000,4.000000


In [6]:
df.isna().sum()

age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64

In [ ]:
df['ca'] = df['ca'].astype(str)
y = df['num']
X = df.drop(columns=['num'])
numerical_feature = X.select_dtypes(include=["int64", "float64"]).columns.to_list()
categorical_feature = X.select_dtypes(exclude=["int64", "float64"]).columns.to_list()
X[categorical_feature] = X[categorical_feature].astype(str)

In [8]:
from numpy import squeeze
def transform_to_biner(x):
    if x != 0:
        x = 1
    return x

y = y.apply(transform_to_biner)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [10]:
ros = RandomOverSampler(random_state=42)
X_train, y_train = ros.fit_resample(X_train, y_train)
print("Proporsi Target Setelah Oversampling:")
print(pd.Series(y_train).value_counts())

Proporsi Target Setelah Oversampling:
num
0    353
1    353
Name: count, dtype: int64


In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), numerical_feature),
        ("categorical", Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_feature)
    ]
)

In [12]:
models = {
    "KNN": (
        KNeighborsClassifier(),
        {
            "train__n_neighbors": [1,2,3,4],
            "train__weights": ["uniform","distance"],
            "train__p": [1,2,3]
        }
    ),

    "LR": (
        LogisticRegression(solver="liblinear", max_iter=5000),
        {
            "train__penalty": ["l1","l2"],
            "train__C": [100,10,1,0.1,0.01,0.001]
        }
    ),

    "SVM": (
        SVC(probability=True),
        {
            "train__kernel": ["linear","rbf"]
        }
    ),

    "NB": (
        BernoulliNB(),
        {
            "train__alpha": [1,0.1,0.01,0.001,0.0001,0.00001]
        }
    ),

    "DT": (
        DecisionTreeClassifier(),
        {
            "train__max_features": ["sqrt","log2",5,10,30],
            "train__max_depth": [2,8,16,32,64,128],
            "train__min_samples_split": [2,4,8,16,24],
            "train__min_samples_leaf": [1,2,5,10,15,30]
        }
    ),

    "RF": (
        RandomForestClassifier(),
        {
            "train__n_estimators": [10,50,100,200,500],
            "train__max_features": ["sqrt","log2",5,10,30],
            "train__max_depth": [2,8,16,32,64,128],
            "train__min_samples_split": [2,4,8,16,24],
            "train__min_samples_leaf": [1,2,5,10,15,30]
        }
    ),

    "RC": (
        RidgeClassifier(),
        {
            "train__alpha": [1,2,3,4,5,6,7,8,9,10]
        }
    ),

    "GP": (
        GaussianProcessClassifier(),
        {
            "train__max_iter_predict": [100,200,300,400,500,600,700,800,900,1000]
        }
    ),

    "LDA": (
        LinearDiscriminantAnalysis(),
        {
            "train__solver": ["lsqr","eigen"],
            "train__shrinkage": ["auto",0.0001,0.001,0.01,0.1,0.5,1]
        }
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

In [13]:
for name, (estimator, param_grid) in models.items():

    print(f"Training {name}...")

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("train", estimator)
    ])

    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring="roc_auc",
        n_jobs=2
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)

    # Untuk AUC
    if hasattr(best_model["train"], "predict_proba"):
        y_proba = best_model.predict_proba(X_test)[:,1]
    else:
        y_proba = best_model.decision_function(X_test)

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_proba),
        "Recall": recall_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Kappa": cohen_kappa_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

    results.append(metrics)

Training KNN...
Training LR...
Training SVM...
Training NB...
Training DT...
Training RF...
Training RC...
Training GP...
Training LDA...


In [14]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("AUC", ascending=False)

print(results_df)

  Model  Accuracy       AUC    Recall  Precision        F1     Kappa       MCC
5    RF  0.829710  0.916506  0.826923   0.865772  0.845902  0.655842  0.656705
2   SVM  0.851449  0.913942  0.852564   0.880795  0.866450  0.699203  0.699673
8   LDA  0.818841  0.909188  0.794872   0.873239  0.832215  0.636306  0.639631
6    RC  0.822464  0.908868  0.807692   0.868966  0.837209  0.642563  0.644641
0   KNN  0.840580  0.906330  0.852564   0.863636  0.858065  0.676264  0.676337
1    LR  0.815217  0.905769  0.788462   0.872340  0.828283  0.629384  0.633157
3    NB  0.822464  0.903152  0.820513   0.859060  0.839344  0.641197  0.642041
7    GP  0.847826  0.896368  0.839744   0.885135  0.861842  0.692748  0.693938
4    DT  0.771739  0.832265  0.769231   0.816327  0.792079  0.539561  0.540733
